# Giai đoạn 5 — Train model (bản cải tiến).

Giai đoạn 5 — Train model (bản cải tiến).
Input : datas/04_split/{X_train_*, y_train_*} + selected_features.json
Output: datas/05_models/{best_casual.pkl, best_registered.pkl, best_cnt.pkl, cv_results_*.csv}
- Prep tách riêng: Linear (Poly+Scale+OneHot) CHỈ cho Ridge/Ridge-log ;
  GLM (Scale+OneHot, không Poly) cho Poisson/Tweedie ; Tree (passthrough+Ordinal)
  cho DecisionTree/RandomForest/HistGradientBoosting
- Ridge-log: TransformedTargetRegressor(log1p/expm1) cho target lệch phải
- HistGradientBoosting (early stopping thật) + Poisson/Tweedie cho count data
- GridSearchCV cho lưới nhỏ, RandomizedSearchCV (n_iter giới hạn) cho RF/HGB
- CV: TimeSeriesSplit(5), scoring neg_MAE
Chạy: python src/05_train.py

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.compose import TransformedTargetRegressor as TTR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import PoissonRegressor, Ridge, TweedieRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.compose import TransformedTargetRegressor as TTR
from numpy import log1p, expm1
# Cấu hình đường dẫn (inline để notebook chạy độc lập, không cần config.py)
ROOT = Path.cwd()
if not (ROOT / "datas").exists() and (ROOT.parent / "datas").exists():
    ROOT = ROOT.parent  # khi kernel chạy từ trong thư mục src/
RAW_CSV = ROOT / "datas" / "hour.csv"
EDA_DIR = ROOT / "datas" / "01_eda"
CLEANED_DIR = ROOT / "datas" / "02_cleaned"
FEATURES_DIR = ROOT / "datas" / "03_features"
SPLIT_DIR = ROOT / "datas" / "04_split"
MODELS_DIR = ROOT / "datas" / "05_models"
EVAL_DIR = ROOT / "datas" / "06_evaluation"
CAT_COLS = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "time_period"]
TEST_SIZE = 0.2
RANDOM_STATE = 42

N_SPLITS = 5
RF_ITER, HGB_ITER = 12, 12  # giới hạn budget tuning cho model đắt


In [2]:
def get_preps(num_cols, cat_cols):
    poly_line = Pipeline([("poly", PolynomialFeatures(include_bias=False)),
                          ("scaler", StandardScaler())])
    prep_line = ColumnTransformer([
        ("num", poly_line, num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
    prep_glm = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
    prep_tree = ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)])
    return prep_line, prep_glm, prep_tree


In [3]:
def get_search_spaces(prep_line, prep_glm, prep_tree):
    ridge_pipe = Pipeline([("prep", prep_line), ("ridge", Ridge())])
    return {
        # Lưới nhỏ -> GridSearch
        "PolyLinear": {"model": ridge_pipe,
                       "search": "grid",
                       "params": {"prep__num__poly__degree": [1, 2],
                                  "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}},
        # Cải tiến: Ridge trên log1p(target) cho count lệch phải
        "RidgeLog": {"model": TTR(regressor=ridge_pipe,
                                  func=log1p, inverse_func=expm1),
                     "search": "grid",
                     "params": {"regressor__prep__num__poly__degree": [1, 2],
                                "regressor__ridge__alpha": [0.1, 1.0, 10.0, 100.0]}},
        "Poisson": {"model": Pipeline([("prep", prep_glm),
                                       ("glm", PoissonRegressor(max_iter=2000))]),
                    "search": "grid",
                    "params": {"glm__alpha": [0.001, 0.01, 0.1, 1.0]}},
        "Tweedie": {"model": Pipeline([("prep", prep_glm),
                                       ("glm", TweedieRegressor(power=1.5, max_iter=2000))]),
                    "search": "grid",
                    "params": {"glm__alpha": [0.001, 0.01, 0.1, 1.0]}},
        "DecisionTree": {"model": Pipeline([("prep", prep_tree),
                                            ("tree", DecisionTreeRegressor(
                                                random_state=RANDOM_STATE))]),
                         "search": "grid",
                         "params": {"tree__max_depth": [3, 5, 8, 12, None],
                                    "tree__min_samples_leaf": [1, 5, 10, 20],
                                    "tree__max_features": ["sqrt", 0.5, 1.0]}},
        # Lưới lớn -> RandomizedSearch để giới hạn chi phí
        "RandomForest": {"model": Pipeline([("prep", prep_tree),
                                            ("rf", RandomForestRegressor(
                                                random_state=RANDOM_STATE, n_jobs=-1))]),
                         "search": "random", "n_iter": RF_ITER,
                         "params": {"rf__n_estimators": randint(100, 400),
                                    "rf__max_depth": [5, 10, 20, None],
                                    "rf__max_features": [0.3, 0.5, "sqrt", 1.0],
                                    "rf__min_samples_leaf": randint(1, 12)}},
        # Cải tiến: Boosting + early stopping thật
        "HistGB": {"model": Pipeline([("prep", prep_tree),
                                      ("hgb", HistGradientBoostingRegressor(
                                          early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=20, random_state=RANDOM_STATE))]),
                   "search": "random", "n_iter": HGB_ITER,
                   "params": {"hgb__max_iter": randint(100, 500),
                              "hgb__max_leaf_nodes": randint(15, 63),
                              "hgb__learning_rate": loguniform(0.02, 0.3),
                              "hgb__l2_regularization": loguniform(1e-3, 10.0)}},
    }


In [4]:
def cv_extra_metrics(model, X, y, kf):
    maes, rms, r2s = [], [], []
    yv = y.values.ravel()
    for tr, va in kf.split(X):
        m = clone(model).fit(X.iloc[tr], yv[tr])
        p = np.clip(m.predict(X.iloc[va]), 0, None)
        maes.append(mean_absolute_error(yv[va], p))
        rms.append(root_mean_squared_error(yv[va], p))
        r2s.append(r2_score(yv[va], p))
    return float(np.mean(maes)), float(np.std(maes)), float(np.mean(rms)), float(np.mean(r2s))


In [5]:
def train_one(X, y, tag):
    sel = json.loads((SPLIT_DIR / "selected_features.json").read_text(encoding="utf-8"))
    if tag == "cnt":
        num_c = sorted(set(sel["num_casual"] + sel["num_registered"]) & set(X.columns))
        cat_c = sorted(set(sel["cat_casual"] + sel["cat_registered"]) & set(X.columns))
    else:
        num_c = [c for c in sel[f"num_{tag}"] if c in X.columns]
        cat_c = [c for c in sel[f"cat_{tag}"] if c in X.columns]
    prep_line, prep_glm, prep_tree = get_preps(num_c, cat_c)
    kf = TimeSeriesSplit(n_splits=N_SPLITS)
    rows, best = [], None
    for name, cfg in get_search_spaces(prep_line, prep_glm, prep_tree).items():
        if cfg["search"] == "grid":
            search = GridSearchCV(cfg["model"], cfg["params"], cv=kf,
                                  scoring="neg_mean_absolute_error", n_jobs=-1)
        else:
            search = RandomizedSearchCV(cfg["model"], cfg["params"],
                                        n_iter=cfg["n_iter"], cv=kf,
                                        scoring="neg_mean_absolute_error",
                                        random_state=RANDOM_STATE, n_jobs=-1)
        search.fit(X, y.values.ravel())
        mae_m, mae_s, rmse_m, r2_m = cv_extra_metrics(search.best_estimator_, X, y, kf)
        rows.append({"model": name, "mae_mean": -search.best_score_,
                     "mae_std": mae_s, "rmse_mean": rmse_m, "r2_mean": r2_m,
                     "best_params": json.dumps(search.best_params_, default=str)})
        if best is None or rows[-1]["mae_mean"] < best[1]:
            best = (search.best_estimator_, rows[-1]["mae_mean"])
        print(f"[{tag}] {name}: MAE={rows[-1]['mae_mean']:.4f} "
              f"RMSE={rmse_m:.2f} R2={r2_m:.4f}")
    pd.DataFrame(rows).sort_values("mae_mean").to_csv(
        MODELS_DIR / f"cv_results_{tag}.csv", index=False)
    joblib.dump(best[0], MODELS_DIR / f"best_{tag}.pkl")
    return best[0]


In [6]:
def main():
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for tag in ["casual", "registered", "cnt"]:
        X = pd.read_csv(SPLIT_DIR / f"X_train_{tag}.csv")
        y = pd.read_csv(SPLIT_DIR / f"y_train_{tag}.csv")
        train_one(X, y, tag)
        print(f"OK best_{tag}.pkl")
    print(f"OK -> {MODELS_DIR}")


In [7]:
if __name__ == "__main__":
    main()


[casual] PolyLinear: MAE=9.6867 RMSE=14.51 R2=0.8966


[casual] RidgeLog: MAE=16.5044 RMSE=36.03 R2=0.2858


[casual] Poisson: MAE=25.3353 RMSE=55.66 R2=-1.8164


[casual] Tweedie: MAE=33.3617 RMSE=79.65 R2=-4.6396


[casual] DecisionTree: MAE=10.2061 RMSE=16.58 R2=0.8633


...

[cnt] Poisson: MAE=63.3243 RMSE=103.66 R2=0.6024


[cnt] Tweedie: MAE=72.2247 RMSE=124.38 R2=0.4188


[cnt] DecisionTree: MAE=41.9393 RMSE=67.42 R2=0.8144


[cnt] RandomForest: MAE=34.4444 RMSE=56.02 R2=0.8690


[cnt] HistGB: MAE=33.4535 RMSE=53.07 R2=0.8814
OK best_cnt.pkl
OK -> /mnt/d/Documents/UIT/HK2/CKIE313/datas/05_models
